In [ ]:
import os
import sqlite3
import asyncio
import json
import urllib.request
import urllib.error
from google.colab import userdata

DB_PATH = "tienda_crochet.db"

# Intentar obtener API KEY desde Secretos de Colab (si existe)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# =============================================================
# AUTO-INGESTA Y CREACIÓN DE BASE DE DATOS RELACIONAL
# =============================================================
def inicializar_base_e_ingesta():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS PATRONES (
        codigo_patron TEXT PRIMARY KEY,
        nombre TEXT NOT NULL,
        talle TEXT NOT NULL,
        segmento_edad TEXT NOT NULL,
        precio_unitario REAL NOT NULL,
        stock INTEGER NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS HILADOS (
        codigo_hilado TEXT PRIMARY KEY,
        marca TEXT NOT NULL,
        linea TEXT NOT NULL,
        grosor TEXT NOT NULL,
        aguja_sugerida TEXT,
        precio_por_kilo REAL NOT NULL,
        stock_kg REAL NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS AGUJAS (
        codigo_aguja TEXT PRIMARY KEY,
        modelo TEXT NOT NULL,
        marca TEXT NOT NULL,
        medida_mm REAL NOT NULL,
        precio_unitario REAL NOT NULL,
        stock_unidades INTEGER NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS RECETA_INSUMOS (
        id_receta INTEGER PRIMARY KEY AUTOINCREMENT,
        codigo_patron TEXT NOT NULL,
        codigo_hilado TEXT NOT NULL,
        codigo_aguja TEXT NOT NULL,
        gramos_necesarios REAL NOT NULL,
        dificultad TEXT NOT NULL,
        FOREIGN KEY (codigo_patron) REFERENCES PATRONES(codigo_patron),
        FOREIGN KEY (codigo_hilado) REFERENCES HILADOS(codigo_hilado),
        FOREIGN KEY (codigo_aguja) REFERENCES AGUJAS(codigo_aguja)
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS TRANSACCIONES (
        id_transaccion TEXT PRIMARY KEY,
        id_cliente TEXT NOT NULL,
        fecha_hora TEXT NOT NULL,
        acepto_cross_selling INTEGER NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS DETALLE_TRANSACCION (
        id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
        id_transaccion TEXT NOT NULL,
        tipo_producto TEXT NOT NULL,
        codigo_producto TEXT NOT NULL,
        cantidad INTEGER NOT NULL,
        FOREIGN KEY (id_transaccion) REFERENCES TRANSACCIONES(id_transaccion)
    );
    """)

    patrones = [
        ('GoPm', 'Gorro Pompón', 'Único', 'Dama', 5000.0, 1000),
        ('BuIn', 'Bufanda Infinita', 'Único', 'Dama', 7500.0, 1000),
        ('PoAl', 'Poncho Alma', '10', 'Niños', 10000.0, 1000),
        ('SnVe', 'Saco Snow Verano', 'M', 'Dama', 15000.0, 1000),
        ('SnIn', 'Saco Snow Invierno', 'M', 'Dama', 15000.0, 1000)
    ]
    cursor.executemany("INSERT OR REPLACE INTO PATRONES VALUES (?,?,?,?,?,?)", patrones)

    hilados = [
        ('Hz', 'Judith M Herz', 'Gordon', 'Semigrueso', '7mm', 38500.0, 7.0),
        ('Cs', 'Cisne', 'Jackard', 'Grueso', '7mm a 9mm', 69000.0, 10.0),
        ('Tr-CS-F', 'Tropezón', 'Cashmilon Sedificado', 'Fino', '4mm', 45000.0, 5.0),
        ('Tr-CS-SG', 'Tropezón', 'Cashmilon Sedificado', 'Semigrueso', '6mm', 49000.0, 40.0),
        ('Tr-HA-83', 'Tropezón', 'Hilo de Algodón con Seda', '8/3', '3mm', 33000.0, 5.0),
        ('Tr-HA-86', 'Tropezón', 'Hilo de Algodón con Seda', '8/6', '4mm', 35000.0, 15.0)
    ]
    cursor.executemany("INSERT OR REPLACE INTO HILADOS VALUES (?,?,?,?,?,?,?)", hilados)

    agujas = [
        ('Al2', 'Aluminio', 'Silver', 2.0, 1300.0, 50),
        ('Al4', 'Aluminio', 'Silver', 4.0, 1300.0, 100),
        ('Al6', 'Aluminio', 'Silver', 6.0, 1750.0, 100),
        ('Al8', 'Aluminio', 'Silver', 8.0, 2250.0, 100),
        ('Mp4', 'Mango Ergonómico de Plástico', 'Genérica', 4.0, 500.0, 50),
        ('Ms6', 'Mango Ergonómico de Silicona', 'Almendra', 6.0, 5000.0, 100)
    ]
    cursor.executemany("INSERT OR REPLACE INTO AGUJAS VALUES (?,?,?,?,?,?)", agujas)

    recetas = [
        ('GoPm', 'Hz', 'Al6', 141.0, 'FÁCIL'),
        ('BuIn', 'Cs', 'Al8', 200.0, 'FÁCIL'),
        ('PoAl', 'Tr-CS-SG', 'Al6', 220.0, 'INTERMEDIO'),
        ('SnVe', 'Tr-HA-86', 'Mp4', 300.0, 'AVANZADO'),
        ('SnIn', 'Tr-CS-SG', 'Ms6', 600.0, 'AVANZADO')
    ]
    cursor.executemany("""
    INSERT INTO RECETA_INSUMOS (codigo_patron, codigo_hilado, codigo_aguja, gramos_necesarios, dificultad)
    VALUES (?,?,?,?,?)
    """, recetas)

    conn.commit()
    conn.close()

def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# =============================================================
# FUNCIONALIDADES DEL SISTEMA (HUs)
# =============================================================
def hu01_gestionar_stock():
    print("\n--- [HU-01] CONSULTA DE INVENTARIO Y STOCK ---")
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("SELECT codigo_hilado, marca, linea, grosor, stock_kg FROM HILADOS")
    filas = cursor.fetchall()
    conn.close()

    print("\nCatalogación de Hilados Disponibles en SQLite:")
    for r in filas:
        print(f"-> Código: {r['codigo_hilado']} | {r['marca']} ({r['linea']}) | Grosor: {r['grosor']} | Stock: {r['stock_kg']} kg")

def hu04_filtrar_patrones():
    print("\n--- [HU-04] FILTRADO DE PATRONES POR DIFICULTAD ---")
    dificultad = input("Ingrese dificultad a buscar (FÁCIL / INTERMEDIO / AVANZADO): ").strip().upper()

    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT P.codigo_patron, P.nombre, P.talle, P.precio_unitario, R.dificultad
        FROM PATRONES P
        JOIN RECETA_INSUMOS R ON P.codigo_patron = R.codigo_patron
        WHERE R.dificultad = ?
    """, (dificultad,))
    filas = cursor.fetchall()
    conn.close()

    if not filas:
        print(f"❌ No se encontraron patrones para el nivel '{dificultad}'.")
    else:
        print(f"\nResultados encontrados para nivel {dificultad}:")
        for r in filas:
            print(f"-> [{r['codigo_patron']}] {r['nombre']} - Talle: {r['talle']} | Precio: ${r['precio_unitario']}")

def obtener_receta_sql(codigo_patron):
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT R.codigo_patron, H.marca AS hilado, H.grosor, A.modelo AS aguja, A.medida_mm, R.gramos_necesarios
        FROM RECETA_INSUMOS R
        JOIN HILADOS H ON R.codigo_hilado = H.codigo_hilado
        JOIN AGUJAS A ON R.codigo_aguja = A.codigo_aguja
        WHERE R.codigo_patron = ?
    """, (codigo_patron,))
    row = cursor.fetchone()
    conn.close()
    return dict(row) if row else None

def llamada_http_gemini(prompt, api_key):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"
    data = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode('utf-8')
    req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})

    with urllib.request.urlopen(req, timeout=4.0) as response:
        res = json.loads(response.read().decode('utf-8'))
        return res['candidates'][0]['content']['parts'][0]['text']

async def hu03_asistente_ia():
    print("\n--- [HU-03] ASISTENTE DE VENTA CRUZADA CON IA (Y FALLBACK 4S) ---")
    codigo_patron = input("Ingrese Código del Patrón (ej: GoPm, BuIn, SnVe): ").strip()
    aguja_mm = input("¿Qué medida de aguja (mm) tenés en casa?: ").strip()
    tiene_hilado = input("¿Tenés hilado en casa? (SI/NO): ").strip().upper()

    receta_base = obtener_receta_sql(codigo_patron)
    if not receta_base:
        print("❌ Error: Código de patrón no encontrado en SQL.")
        return

    prompt = f"Actúa como asistente de crochet. Patrón: {codigo_patron}. Receta oficial: Hilado {receta_base['hilado']} ({receta_base['grosor']}), Aguja {receta_base['medida_mm']}mm. El cliente tiene aguja de {aguja_mm}mm y {'tiene' if tiene_hilado=='SI' else 'no tiene'} hilado. Recomienda en 2 oraciones breves qué comprar."

    print("\n[Consultando al Asistente de IA (Timeout síncrono: 4.0s)...]")

    try:
        if not GEMINI_API_KEY:
            raise ValueError("Sin API Key configurada. Activando fallback de resiliencia.")

        respuesta_ia = await asyncio.to_thread(llamada_http_gemini, prompt, GEMINI_API_KEY)
        print("\n✅ [RESPUESTA DE IA GENERATIVA EN VIVO]:")
        print(respuesta_ia)

    except (asyncio.TimeoutError, Exception) as e:
        print(f"\n⚠️ [ALERTA DE RESILIENCIA / TIMEOUT > 4s]: {e}")
        print("Activando ejecutarFallback(). Cargando receta predeterminada desde SQL:")
        print(f"-> Recomendar Kit Base: {receta_base['gramos_necesarios']}g de Hilado {receta_base['hilado']} + Aguja {receta_base['medida_mm']}mm.")

def hu02_procesar_compra():
    print("\n--- [HU-02] REGISTRO DE COMPRA EN CHECKOUT ---")
    id_cliente = input("ID de Cliente (ej: CLI-001): ").strip()
    codigo_patron = input("Código de Patrón a comprar: ").strip()
    acepto_kit = input("¿Aceptó el kit del asistente? (SI/NO): ").strip().upper()

    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM TRANSACCIONES")
    count = cursor.fetchone()[0] + 1
    tid = f"TRX-{count:04d}"

    cursor.execute("""
        INSERT INTO TRANSACCIONES (id_transaccion, id_cliente, fecha_hora, acepto_cross_selling)
        VALUES (?, ?, datetime('now'), ?)
    """, (tid, id_cliente, 1 if acepto_kit == 'SI' else 0))

    cursor.execute("""
        INSERT INTO DETALLE_TRANSACCION (id_transaccion, tipo_producto, codigo_producto, cantidad)
        VALUES (?, 'PATRON', ?, 1)
    """, (tid, codigo_patron))

    conn.commit()
    conn.close()
    print(f"✅ ¡Compra guardada exitosamente en SQL! ID: {tid}")

# =============================================================
# INICIALIZACIÓN AUTOMÁTICA Y MENÚ
# =============================================================
async def ejecutar_demo():
    # Asegura que la BD y tablas existan siempre antes de abrir el menú
    inicializar_base_e_ingesta()

    while True:
        print("\n=============================================")
        print("   PROTOTIPO SISTEMA TEJIDOS CON IA (DEMO)   ")
        print("=============================================")
        print("1. [HU-01] Ver stock de inventario")
        print("2. [HU-04] Filtrar patrones por dificultad")
        print("3. [HU-03] Probar Asistente de IA (con Fallback 4s)")
        print("4. [HU-02] Procesar compra en checkout")
        print("5. Salir")

        op = input("\nSeleccione opción (1-5): ").strip()
        if op == "1":
            hu01_gestionar_stock()
        elif op == "2":
            hu04_filtrar_patrones()
        elif op == "3":
            await hu03_asistente_ia()
        elif op == "4":
            hu02_procesar_compra()
        elif op == "5":
            print("\nDemo finalizada.")
            break
        else:
            print("Opción inválida.")

await ejecutar_demo()


   PROTOTIPO SISTEMA TEJIDOS CON IA (DEMO)   
1. [HU-01] Ver stock de inventario
2. [HU-04] Filtrar patrones por dificultad
3. [HU-03] Probar Asistente de IA (con Fallback 4s)
4. [HU-02] Procesar compra en checkout
5. Salir

Seleccione opción (1-5): 1

--- [HU-01] CONSULTA DE INVENTARIO Y STOCK ---

Catalogación de Hilados Disponibles en SQLite:
-> Código: Hz | Judith M Herz (Gordon) | Grosor: Semigrueso | Stock: 7.0 kg
-> Código: Cs | Cisne (Jackard) | Grosor: Grueso | Stock: 10.0 kg
-> Código: Tr-CS-F | Tropezón (Cashmilon Sedificado) | Grosor: Fino | Stock: 5.0 kg
-> Código: Tr-CS-SG | Tropezón (Cashmilon Sedificado) | Grosor: Semigrueso | Stock: 40.0 kg
-> Código: Tr-HA-83 | Tropezón (Hilo de Algodón con Seda) | Grosor: 8/3 | Stock: 5.0 kg
-> Código: Tr-HA-86 | Tropezón (Hilo de Algodón con Seda) | Grosor: 8/6 | Stock: 15.0 kg

   PROTOTIPO SISTEMA TEJIDOS CON IA (DEMO)   
1. [HU-01] Ver stock de inventario
2. [HU-04] Filtrar patrones por dificultad
3. [HU-03] Probar Asistente de 